# Notebook 2: The Neo4j Backend

SGET uses a **Neo4j graph database** as its live backend. Scene graphs are loaded from JSON into Neo4j, and all edits happen against the database. This notebook walks through the pipeline: bulk loading with **heracles**, querying with **Cypher**, and single-node operations with SGET's **CRUD layer**.

## Prerequisites
- Neo4j running on localhost:7687 (Docker: `docker run -d -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/neo4j_pw neo4j:5.25.1`)
- Virtual environment activated with SGET, spark_dsg, and heracles installed

## 1. Connecting to Neo4j

The **heracles** library provides `Neo4jWrapper`, a thin wrapper around the Neo4j Python driver. It handles connection management and query execution.

In [ ]:
from heracles.query_interface import Neo4jWrapper

db = Neo4jWrapper(
    db_uri="neo4j://127.0.0.1:7687",
    db_auth=("neo4j", "neo4j_pw"),
    atomic_queries=True,  # Each query auto-commits (no manual transactions)
)
db.connect()

# Verify the connection with a simple query.
records, _, _ = db.execute("RETURN 'Connected!' AS status")
print(records[0]["status"])

## 2. Bulk Loading with heracles

Heracles converts an entire spark_dsg scene graph into Neo4j in one shot using a **property-presence** approach — it stores whatever attributes each node has, without hardcoding assumptions about specific types.

The process:
1. **`initialize_db(db)`** — clears the database and creates indexes on `nodeSymbol` for each node label
2. **`spark_dsg_to_db(G, db)`** — creates all nodes and edges, layer by layer

Heracles requires two metadata entries on the DSG object before loading:
- `"labelspace"` — maps semantic label IDs to human-readable names (e.g., `{34: "box"}`)
- `"LayerIdToHeraclesLayerStr"` — maps spark_dsg layer IDs to Neo4j node labels

Every node gets an **`attr_type`** property storing the Python class name (e.g., `"ObjectNodeAttributes"`, `"TravNodeAttributes"`). This is used for reconstruction when exporting back to spark_dsg.

Common properties stored per type:

| Neo4j Label | Possible attr_types | Common Properties |
|-------------|---------------------|-------------------|
| `Object` | ObjectNodeAttributes, KhronosObjectAttributes | nodeSymbol, attr_type, center, class, name, bbox_*, color_*, registered |
| `Place` | PlaceNodeAttributes | nodeSymbol, attr_type, center, distance |
| `MeshPlace` | Place2dNodeAttributes, TravNodeAttributes | nodeSymbol, attr_type, center, class (Place2d) or radii/states (Trav) |
| `Room` | RoomNodeAttributes | nodeSymbol, attr_type, center, class |
| `Building` | NodeAttributes | nodeSymbol, attr_type, center |

**TIP**: Not all nodes have the same properties. To discover what a node has:
```cypher
MATCH (n:MeshPlace) RETURN keys(n) AS props LIMIT 1
MATCH (n:MeshPlace) RETURN DISTINCT n.attr_type
```

In [ ]:
import os

import spark_dsg
from heracles import constants
from heracles.graph_interface import initialize_db, spark_dsg_to_db
from heracles.utils import get_labelspace

# Load the DSG from JSON (same file as Notebook 1).
DSG_PATH = os.path.expanduser(
    "~/software/mit/sget/heracles/heracles/examples/scene_graphs/example_dsg.json"
)
G = spark_dsg.DynamicSceneGraph.load(DSG_PATH)

# Load labelspaces from heracles' bundled YAML files.
# These map integer semantic IDs to human-readable class names.
object_labels = get_labelspace("ade20k_mit_label_space.yaml")  # {id: name}
room_labels = get_labelspace("b45_label_space.yaml")

print(f"Object labelspace: {len(object_labels)} classes")
print(f"  Examples: {dict(list(object_labels.items())[:5])}")
print(f"Room labelspace: {room_labels}")

In [ ]:
# Inject the metadata heracles needs for bulk load.
G.metadata.add({"labelspace": object_labels})
G.metadata.add({"room_labelspace": room_labels})
G.metadata.add(
    {
        "LayerIdToHeraclesLayerStr": {
            2: "Object",
            3: "Place",
            4: "Room",
            5: "Building",
            20: "MeshPlace",
            "3[1]": "MeshPlace",
        }
    }
)

# Clear the database and load the scene graph.
initialize_db(db)
spark_dsg_to_db(G, db)

# Verify: count nodes per label.
for label in [
    constants.OBJECTS,
    constants.PLACES,
    constants.MESH_PLACES,
    constants.ROOMS,
    constants.BUILDINGS,
]:
    records, _, _ = db.execute(f"MATCH (n:{label}) RETURN count(n) AS cnt")
    print(f"  {label}: {records[0]['cnt']} nodes")

## 3. Querying with Cypher

Neo4j uses **Cypher** as its query language. Here are the patterns you'll see most often in the SGET codebase. You don't need to master Cypher — just enough to read the queries in `neo4j_crud.py`.

In [ ]:
# Find all distinct object classes and their counts.
results = db.query(
    "MATCH (n:Object) RETURN DISTINCT n.class AS class, count(*) AS count ORDER BY count DESC"
)
for r in results:
    print(f"  {r['class']}: {r['count']}")

In [ ]:
# Look up a specific node by its nodeSymbol.
results = db.query("MATCH (n:Object {nodeSymbol: 'O0'}) RETURN n")
print("Object O0:", results[0]["n"])

In [ ]:
# Traverse the hierarchy: find all Objects contained (transitively) in Room R1.
# The * after CONTAINS means "follow CONTAINS edges recursively".
results = db.query("""
    MATCH (r:Room {nodeSymbol: 'R1'})-[:CONTAINS*]->(o:Object)
    RETURN o.nodeSymbol AS obj, o.class AS class
""")
print(f"Objects in Room R1: {len(results)}")
for r in results[:5]:
    print(f"  {r['obj']}: {r['class']}")
if len(results) > 5:
    print(f"  ... and {len(results) - 5} more")

## 4. The SGET CRUD Layer

Heracles only provides **bulk** operations (load an entire graph, export an entire graph). For interactive editing, SGET needs to create, read, update, and delete **individual** nodes and edges. That's what `sget.backend.neo4j_crud` provides.

Each function takes a `Neo4jWrapper` as its first argument and executes targeted Cypher queries. Key design choices:
- **Per-layer CREATE templates**: each layer has different properties (Objects have bounding boxes, Places don't), so creation uses explicit templates that match heracles' schema.
- **Generic UPDATE**: builds `SET` clauses dynamically. Point3D fields (center, bbox_center, bbox_dim) are wrapped in Neo4j's `point()` function; scalars are set directly.
- **Edge type inference**: `determine_edge_type()` figures out whether an edge is intralayer (e.g., `PLACE_CONNECTED`) or interlayer (`CONTAINS`) based on the source/target labels.

In [ ]:
from sget.backend.neo4j_crud import (
    create_edge,
    create_node,
    delete_node,
    determine_edge_type,
    get_node,
    update_node,
)

# CREATE — add a new Object node.
create_node(
    db,
    constants.OBJECTS,
    "o(99)",
    {
        "pos_x": 5.0,
        "pos_y": 5.0,
        "pos_z": 0.0,
        "bbox_x": 5.0,
        "bbox_y": 5.0,
        "bbox_z": 0.0,
        "bbox_l": 1.0,
        "bbox_w": 1.0,
        "bbox_h": 1.0,
        "class": "box",
        "name": "tutorial_box",
    },
)

# READ — fetch it back.
node = get_node(db, constants.OBJECTS, "o(99)")
print("Created node:", node)

In [ ]:
# UPDATE — change its name and position.
update_node(
    db,
    constants.OBJECTS,
    "o(99)",
    {
        "name": "renamed_box",
        "center": [10.0, 10.0, 0.0],  # Point3D fields use [x, y, z] lists
    },
)

node = get_node(db, constants.OBJECTS, "o(99)")
print(f"Updated: name={node['name']}, center={node['center']}")

In [ ]:
# EDGES — edge type is inferred from the labels of the two endpoints.
print("Same layer (Object-Object):", determine_edge_type(constants.OBJECTS, constants.OBJECTS))
print("Cross layer (Room-Place):  ", determine_edge_type(constants.ROOMS, constants.PLACES))

# Create a CONTAINS edge from Room R1 to our new object.
create_edge(db, constants.ROOMS, "R1", constants.OBJECTS, "o(99)")

# Verify it exists.
results = db.query("""
    MATCH (r:Room {nodeSymbol: 'R1'})-[:CONTAINS]->(o:Object {nodeSymbol: 'o(99)'})
    RETURN o.name AS name
""")
print(f"\nEdge created: R1 -[:CONTAINS]-> o(99), name={results[0]['name']}")

In [ ]:
# DELETE — clean up our tutorial node.
# delete_node uses DETACH DELETE, which removes the node AND all its edges.
delete_node(db, constants.OBJECTS, "o(99)")

node = get_node(db, constants.OBJECTS, "o(99)")
print(f"After delete: {node}")  # Should be None

## 5. Round-trip: Neo4j → spark_dsg → JSON

Heracles can also reconstruct a spark_dsg graph from the database. This is how SGET's "Save As JSON" works — it exports the current Neo4j state back to a file.

In [ ]:
import tempfile

from heracles.graph_interface import db_to_spark_dsg

# Reconstruct the DSG from Neo4j.
# This needs the layer ID→name mapping and the labelspace dicts.
layer_map = {2: "Object", 3: "Place", 4: "Room", 5: "Building", 20: "MeshPlace"}
label_to_id = {v: k for k, v in object_labels.items()}  # {"box": 34, ...}
room_label_to_id = {v: k for k, v in room_labels.items()}

reconstructed = db_to_spark_dsg(db, layer_map, label_to_id, room_label_to_id)

print(f"Reconstructed: {reconstructed.num_nodes()} nodes, {reconstructed.num_edges()} edges")

# Save to a temporary file.
with tempfile.NamedTemporaryFile(suffix=".json", delete=False) as f:
    reconstructed.save(f.name)
    print(f"Saved to: {f.name}")
    print(f"File size: {os.path.getsize(f.name)} bytes")

In [ ]:
# Clean up.
db.close()
print("Disconnected from Neo4j.")

## Summary

You've learned:
- **heracles** bulk-loads a spark_dsg scene graph into Neo4j with `initialize_db()` + `spark_dsg_to_db()`
- Neo4j stores nodes with labels (Object, Room, etc.), `nodeSymbol` indexes, and `Point3D` coordinates
- **Cypher** queries can filter by label, traverse `CONTAINS` edges transitively, and match by property
- SGET's **CRUD layer** (`neo4j_crud`) extends heracles with single-node create/read/update/delete
- The full pipeline round-trips: JSON → spark_dsg → Neo4j → spark_dsg → JSON

**Next notebook**: [03_sget_architecture.ipynb](03_sget_architecture.ipynb) — how the SGET application is structured and how to extend it.